# EDA: graficos, pruebas estadisticas y regresion

Notebook de demostracion para importar y usar las clases del modulo `scripts`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris

from scripts import (
    GraficosCuantitativos,
    GraficosCualitativos,
    TestEstadisticos,
    RegresionLineal,
    RegresionLogistica,
)

sns.set(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
})
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df.head()

## 1) Graficos cuantitativos

In [ ]:
g_cuant = GraficosCuantitativos(df)

g_cuant.heatmap_correlacion()
plt.show()

g_cuant.scatter("sepal_length", "petal_length", hue="species")
plt.show()

g_cuant.boxplot(x="species", y="petal_width")
plt.show()

g_cuant.violin(x="species", y="sepal_width")
plt.show()

g_cuant.lineplot(x="sepal_length", y="petal_length", hue="species")
plt.show()

g_cuant.regplot("sepal_length", "petal_length")
plt.show()

g_cuant.pairplot(hue="species")
plt.show()

## 2) Graficos cualitativos

In [ ]:
g_cual = GraficosCualitativos(df, cols=["species"])

g_cual.countplot("species")
plt.show()

g_cual.bar_frecuencias("species")
plt.show()

g_cual.pie("species")
plt.show()

## 3) Pruebas estadisticas

In [ ]:
tests = TestEstadisticos(df)

print("ANOVA (petal_length por especie):")
display(tests.anova("petal_length", "species"))

print("\nMANOVA:")
display(tests.manova("sepal_length + sepal_width ~ species"))

print("\nt de Student (setosa vs versicolor):")
display(pd.Series(tests.ttest_student(
    "petal_length", "species", "setosa", "versicolor"
)))

df["larga"] = (df["petal_length"] > df["petal_length"].median()).map({True: "si", False: "no"})
print("\nChi-cuadrado (species vs larga):")
chi = tests.chi_cuadrado("species", "larga")
display(pd.Series({k: v for k, v in chi.items() if k not in ("tabla_observada", "tabla_esperada")}))
display(chi["tabla_observada"])

## 4) Regresion lineal multiple

In [ ]:
reg_lin = RegresionLineal(
    "petal_length ~ sepal_length + sepal_width + C(species)",
    df,
).ajustar()

print(reg_lin.resumen())
display(reg_lin.coeficientes())
display(reg_lin.bondad_ajuste())

reg_lin.grafico_regresion("sepal_length", "petal_length")
plt.show()

reg_lin.grafico_predichos_vs_observados()
plt.show()

reg_lin.diagnostico_residuos()
plt.show()

## 5) Regresion logistica

In [ ]:
df["es_setosa"] = (df["species"] == "setosa").astype(int)

reg_log = RegresionLogistica(
    "es_setosa ~ sepal_length + petal_length",
    df,
).ajustar()

print(reg_log.resumen())
display(reg_log.coeficientes())
display(reg_log.bondad_ajuste())
display(reg_log.matriz_confusion_basica())

reg_log.curva_roc_aprox("petal_length")
plt.show()